# 03 · Robustness of engineered features

**Question:** Is the improvement from `Title` and `Title + Family` consistent when we change the CV splits?

Compare the same Gradient Boosting pipeline across **5 folds × 5 repeats (25 validation scores)**. All feature sets use identical train/validation indices, enabling paired comparisons. This tests robustness; repeated folds are correlated and are **not** 25 independent experiments.

**Original experiment:** `08_testrobustness.ipynb`.

## 1. Import and load data

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

In [ ]:
from pathlib import Path

# Works when running Jupyter from either the repository root or notebooks/.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_dir = next(
    (folder for folder in candidates if (folder / 'data' / 'train.csv').exists()),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        'data/train.csv not found. Download train.csv and test.csv from '
        'https://www.kaggle.com/competitions/titanic/data and put them in data/.'
    )

In [ ]:
sys.path.insert(0, str(project_dir / 'src'))
from feature_engineering import create_experiment_pipeline

train = pd.read_csv(project_dir / 'data' / 'train.csv')
X = train.drop(columns='Survived')
y = train['Survived']

experiments = ['Baseline', 'Title', 'Title + Family']
cv = RepeatedStratifiedKFold(
    n_splits=5, n_repeats=5, random_state=42,
)
splits = list(cv.split(X, y))

## 2. Repeated cross-validation

In [ ]:
all_scores = {}
for name in experiments:
    pipeline = create_experiment_pipeline(
        feature_set=name,
        classifier=GradientBoostingClassifier(random_state=42),
    )
    scores = cross_val_score(
        pipeline, X, y, cv=splits, scoring='accuracy', n_jobs=1,
    )
    all_scores[name] = scores
    print(f'{name:16} mean={scores.mean():.4f} std={scores.std():.4f}')

summary = pd.DataFrame([
    {'Feature group': name, 'Mean CV': s.mean(), 'Std': s.std(),
     'Min': s.min(), 'Max': s.max()}
    for name, s in all_scores.items()
]).sort_values('Mean CV', ascending=False)
display(summary.round(4))

## 3. Paired comparisons on identical splits

In [ ]:
comparisons = [
    ('Title', 'Baseline'),
    ('Title + Family', 'Baseline'),
    ('Title + Family', 'Title'),
]
paired = []
for new, reference in comparisons:
    delta = all_scores[new] - all_scores[reference]
    paired.append({
        'Comparison': f'{new} vs {reference}',
        'Mean change (pp)': delta.mean() * 100,
        'Std of change (pp)': delta.std() * 100,
        'Better folds': int((delta > 0).sum()),
        'Tied folds': int((delta == 0).sum()),
        'Worse folds': int((delta < 0).sum()),
    })
display(pd.DataFrame(paired).round(3))

## 4. Visualize variation between splits

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.boxplot([all_scores[name] for name in experiments])
ax.set_xticks(range(1, len(experiments) + 1), labels=experiments)
ax.set_ylabel('Validation accuracy')
ax.set_title('Repeated stratified 5-fold CV (5 repeats)')
ax.grid(axis='y', alpha=0.25)
plt.show()

scores_df = pd.DataFrame(all_scores)
scores_df['Repeat'] = np.repeat(np.arange(1, 6), 5)
repeat_means = scores_df.groupby('Repeat').mean(numeric_only=True)
display(repeat_means.round(4))

### Outcome and Kaggle submission

**Recorded original repeated CV means:** Baseline **0.8267**, Title **0.8269**, Title + Family **0.8285**. The additional engineered features showed only a small average improvement across 25 splits (Title + Family vs baseline: **+0.179 percentage points**), with 11 improved splits, 10 worse, and 4 ties.

The Title + Family submission scored **0.76315** on the Kaggle public leaderboard, below the simpler baseline submission (**0.79665**). Public leaderboard accuracy and CV accuracy measure different sample sets; this mismatch is a reason to favor a robust, simple baseline rather than claim the features generalize better.

## 5. Reproduce the Title + Family submission (optional)

In [ ]:
test = pd.read_csv(project_dir / 'data' / 'test.csv')
model = create_experiment_pipeline(
    'Title + Family', GradientBoostingClassifier(random_state=42)
)
model.fit(X, y)
predictions = model.predict(test).astype(int)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions,
})
assert len(submission) == len(test)
assert submission['PassengerId'].is_unique
assert submission['Survived'].isin([0, 1]).all()
assert not submission.isna().any().any()

submission_dir = project_dir / 'submissions'
submission_dir.mkdir(exist_ok=True)
path = submission_dir / 'submission_gb_title_family.csv'
submission.to_csv(path, index=False)
print(f'Saved {path.relative_to(project_dir)}')
display(submission.head())